# Caraxis — Fine-tune Llama 3.2 3B on Google Colab

QLoRA + SFT training for the Caraxis security analyst model.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Accept the Llama 3.2 license on Hugging Face: [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
3. Upload `train.jsonl` and `val.jsonl` from `data/caraxis_analyst_v1/processed/` to Google Drive

**Model:** `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`

## 1. Install dependencies

In [ ]:
%%capture
import os, re, torch

v = re.match(r"^[\d]+\.[\d]+", str(torch.__version__)).group(0)
print("Torch version:", v)

# Fix corrupted transformers install if present (~ransformers warning)
!pip uninstall -y transformers >/dev/null 2>&1 || true
!rm -rf /usr/local/lib/python3.13/dist-packages/~ransformers* 2>/dev/null || true

if v in {"2.9"}:
    xformers = "xformers==0.0.33.post1"
elif v in {"2.8"}:
    xformers = "xformers==0.0.32.post2"
elif v in {"2.7", "2.6"}:
    xformers = "xformers==0.0.29.post3"
elif v in {"2.5"}:
    xformers = "xformers==0.0.28.post2"
else:
    # Torch 2.10+ / 2.11+ on current Colab: keep existing xformers wheel
    xformers = "xformers"

!pip install --no-deps bitsandbytes accelerate {xformers} "peft==0.17.1" triton cut_cross_entropy
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps "trl==0.22.2" "transformers==4.57.1"
!pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

print("Install complete")

## 2. Verify installation

In [ ]:
import torch
import peft, transformers, trl

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)

from unsloth import FastLanguageModel
print("Unsloth import OK")

## 3. Hugging Face login

In [ ]:
from huggingface_hub import login
login()  # paste your HF token (needs Llama 3.2 license accepted)

## 4. Load dataset

Update `TRAIN_PATH` and `VAL_PATH` to where you uploaded the JSONL files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE THESE PATHS
TRAIN_PATH = "/content/drive/MyDrive/caraxis/train.jsonl"
VAL_PATH   = "/content/drive/MyDrive/caraxis/val.jsonl"

import os
assert os.path.exists(TRAIN_PATH), f"Missing: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"Missing: {VAL_PATH}"
print("Dataset files found")

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})
print(raw)
print("\nExample messages:")
print(raw["train"][0]["messages"][1]["content"][:300], "...")

## 5. Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # use 1024 if OOM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## 6. Format dataset for SFT

In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = raw.map(format_chat, remove_columns=raw["train"].column_names)
print(dataset["train"][0]["text"][:500])

## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/caraxis_lora_output"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        seed=42,
    ),
)

trainer.train()

## 8. Save adapter

In [ ]:
ADAPTER_DIR = "/content/caraxis_lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved to", ADAPTER_DIR)

## 9. Quick inference test

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Investigate this alert:

<untrusted_data>
{
  "alert_id": "ALRT-TEST-001",
  "severity": "high",
  "title": "Multiple failed SSH authentications",
  "source_ip": "185.220.101.47",
  "host": "prod-web-02",
  "failed_attempts": 847,
  "targeted_accounts": ["root", "admin", "deploy"]
}
</untrusted_data>"""

messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

## 10. Export GGUF for Ollama (optional)

In [ ]:
GGUF_DIR = "/content/caraxis_gguf"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print("GGUF saved to", GGUF_DIR)

## 11. Download adapter to your machine

In [ ]:
!zip -r /content/caraxis_lora_adapter.zip /content/caraxis_lora_adapter

from google.colab import files
files.download("/content/caraxis_lora_adapter.zip")

## 12. Copy adapter to Google Drive (recommended)

In [ ]:
import shutil
DRIVE_OUT = "/content/drive/MyDrive/caraxis/caraxis_lora_adapter"
shutil.copytree(ADAPTER_DIR, DRIVE_OUT, dirs_exist_ok=True)
print("Copied adapter to", DRIVE_OUT)

## Next steps (on your local machine)

1. Score base vs fine-tuned on `evals/caraxis_bench_v0.jsonl`
2. Run `python evals/score.py` and `python evals/report.py`
3. If improved → proceed to Phase 2 (RAG + tools)